In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score,roc_curve

In [2]:
df = pd.read_csv("../Data/cs-test.csv")
pd.set_option('float_format','{:3f}'.format)
X = df.drop('SeriousDlqin2yrs', axis=1)
y = df['SeriousDlqin2yrs']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [3]:
pd.describe()

AttributeError: module 'pandas' has no attribute 'describe'

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 101503 entries, 0 to 101502
Data columns (total 12 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   Unnamed: 0                            101503 non-null  int64  
 1   SeriousDlqin2yrs                      0 non-null       float64
 2   RevolvingUtilizationOfUnsecuredLines  101503 non-null  float64
 3   age                                   101503 non-null  int64  
 4   NumberOfTime30-59DaysPastDueNotWorse  101503 non-null  int64  
 5   DebtRatio                             101503 non-null  float64
 6   MonthlyIncome                         81400 non-null   float64
 7   NumberOfOpenCreditLinesAndLoans       101503 non-null  int64  
 8   NumberOfTimes90DaysLate               101503 non-null  int64  
 9   NumberRealEstateLoansOrLines          101503 non-null  int64  
 10  NumberOfTime60-89DaysPastDueNotWorse  101503 non-null  int64  
 11  NumberOfDep

In [9]:
#Median
income_median = X_train['MonthlyIncome'].median()

income_median = X_train['MonthlyIncome'].median()
temp_income = X_train['MonthlyIncome'].fillna(income_median)

#Cap

income_cap = temp_income.quantile(0.99)

In [10]:
# feature engineering
def transform(data_frame, income_median, income_cap):

    df = data_frame.copy()


    df['IncomeInformed'] = np.where(df['MonthlyIncome'].isnull(), 0, 1)


    df['MonthlyIncome_imp'] = df['MonthlyIncome'].fillna(income_median)


    df['MonthlyIncome_w'] = df['MonthlyIncome_imp'].clip(upper=income_cap)


    df['MonthlyIncome_log'] = 0.0

    mask = df['MonthlyIncome_w'] > 0

    df.loc[mask, 'MonthlyIncome_log'] = np.log(df.loc[mask, 'MonthlyIncome_w'])

    df['IncomexInformed'] = (df['MonthlyIncome_log']*df['IncomeInformed'])

    df['DebtRatio_Caped'] = np.where(df['DebtRatio']>1,1,df['DebtRatio'])

    df['NumberOfDependents_filtered'] = df['NumberOfDependents'].fillna(0)



    return df

In [11]:
#Transformation Train Test
X_train_prep = transform(X_train, income_median, income_cap)
X_test_prep  = transform(X_test, income_median, income_cap)